In [1]:
import json
from sklearn.metrics import cohen_kappa_score
# Round 1
annotated_ann1 = "data_ann1.json"
annotated_ann2 = "data_ann2.json"
with open(annotated_ann1, 'r', encoding='utf-8') as file:
    ann1 = json.load(file)


with open(annotated_ann2, 'r', encoding='utf-8') as file:
    ann2 = json.load(file)

In [2]:
TYPES = ["Lexical", "Syntactic", "Semantic", "Vagueness", "Incompleteness", "Referential"]

# ann1gn by id instead of trusting file order
ann1_by_id = {r["id"]: r for r in ann1}
ann2_by_id = {r["id"]: r for r in ann2}
ids = sorted(set(ann1_by_id) & set(ann2_by_id))
print(f"{len(ids)} shared ids "
      f"(ann1 only: {sorted(set(ann1_by_id) - set(ann2_by_id))}, "
      f"ann2 only: {sorted(set(ann2_by_id) - set(ann1_by_id))})")

for t in TYPES:
    a = [int(ann1_by_id[i][t]) for i in ids]
    w = [int(ann2_by_id[i][t]) for i in ids]
    agree = sum(x == y for x, y in zip(a, w)) / len(ids)
    if len(set(a)) == 1 and len(set(w)) == 1 and a[0] == w[0]:
        k = float("nan")  # both annotators used one label only — kappa undefined
    else:
        k = cohen_kappa_score(a, w, labels=[0, 1])
    print(f"{t:<15} kappa={k:6.3f}  agreement={agree:.2%}  "
          f"ann1_pos={sum(a):2d}  ann2_pos={sum(w):2d}")

148 shared ids (ann1 only: [], ann2 only: [])
Lexical         kappa= 0.206  agreement=95.27%  ann1_pos= 7  ann2_pos= 2
Syntactic       kappa= 0.588  agreement=97.30%  ann1_pos= 7  ann2_pos= 3
Semantic        kappa= 0.664  agreement=99.32%  ann1_pos= 2  ann2_pos= 1
Vagueness       kappa= 0.725  agreement=87.84%  ann1_pos=41  ann2_pos=55
Incompleteness  kappa= 0.350  agreement=88.51%  ann1_pos=15  ann2_pos=14
Referential     kappa= 0.718  agreement=97.97%  ann1_pos= 7  ann2_pos= 4


In [3]:
disagreements = {t: [i for i in ids if ann1_by_id[i][t] != ann2_by_id[i][t]] for t in TYPES}
disagreements

{'Lexical': [53, 54, 55, 56, 57, 69, 70],
 'Syntactic': [19, 50, 104, 129],
 'Semantic': [75],
 'Vagueness': [4,
  5,
  6,
  20,
  28,
  37,
  41,
  42,
  43,
  61,
  62,
  94,
  95,
  131,
  132,
  133,
  134,
  136],
 'Incompleteness': [14,
  22,
  23,
  28,
  29,
  40,
  53,
  54,
  55,
  56,
  57,
  65,
  86,
  89,
  90,
  118,
  136],
 'Referential': [71, 111, 112]}

In [4]:
for i in ids:
    diffs = [t for t in TYPES if bool(ann1_by_id[i][t]) != bool(ann2_by_id[i][t])]
    if not diffs:
        continue
    print(f"id: {i}")
    print(f"requirement: {ann1_by_id[i]['description']}")
    for t in diffs:
        who = "ann1" if ann1_by_id[i][t] else "ann2"
        print(f"  - {t}: True by {who}")
    for name, rec in (("ann1", ann1_by_id[i]), ("ann2", ann2_by_id[i])):
        note = (rec.get("notes") or rec.get("note") or "").strip()
        if note:
            print(f"  note ({name}): {note}")
    print()

id: 4
requirement: The system shall implement procedures for safeguarding passwords.
  - Vagueness: True by ann2

id: 5
requirement: The system shall support implementing policies and procedures to address security incidents.
  - Vagueness: True by ann2

id: 6
requirement: The system shall identify and respond to suspected or known security incidents.
  - Vagueness: True by ann2

id: 14
requirement: The system shall support procedures for obtaining necessary electronic protected health information (ePHI) during an emergency.
  - Incompleteness: True by ann2

id: 19
requirement: The system shall implement electronic mechanisms to corroborate that electronic protected health information (ePHI) has not been altered or destroyed in an unauthorized manner.
  - Syntactic: True by ann1
  note (ann1): not [alteered and destroyed]

id: 20
requirement: The system shall implement procedures to verify that a person or entity seeking access to electronic protected health information (ePHI) is the o